# RetailPulse — Data Validation

## 1. Imports

In [ ]:
import pandas as pd
import numpy as np
import great_expectations as gx
import warnings

warnings.filterwarnings('ignore')

print('great_expectations version:', gx.__version__)

## 2. Load Datasets

In [ ]:
df_sales     = pd.read_csv('../data/processed/retail_clean.csv', low_memory=False)
df_churn     = pd.read_csv('../data/raw/online_retail_customer_churn.csv')
df_inventory = pd.read_csv('../data/raw/retail_store_inventory.csv')
df_rfm       = pd.read_csv('../data/processed/rfm_scores.csv')

print('Sales:    ', df_sales.shape)
print('Churn:    ', df_churn.shape)
print('Inventory:', df_inventory.shape)
print('RFM:      ', df_rfm.shape)

## 3. Setup Great Expectations Context

In [ ]:
# ephemeral context - no file system setup needed
context = gx.get_context(mode='ephemeral')
print('context ready:', type(context).__name__)

## 4. Validate Sales Dataset

In [ ]:
ds_sales = context.data_sources.add_pandas('sales_source')
da_sales = ds_sales.add_dataframe_asset('sales_asset')
bd_sales = da_sales.add_batch_definition_whole_dataframe('sales_batch')

suite_sales = context.suites.add(gx.ExpectationSuite(name='sales_suite'))

suite_sales.add_expectation(gx.expectations.ExpectColumnValuesToNotBeNull(column='Invoice'))
suite_sales.add_expectation(gx.expectations.ExpectColumnValuesToNotBeNull(column='Customer ID'))
suite_sales.add_expectation(gx.expectations.ExpectColumnValuesToNotBeNull(column='InvoiceDate'))
suite_sales.add_expectation(gx.expectations.ExpectColumnValuesToBeBetween(column='Quantity', min_value=1))
suite_sales.add_expectation(gx.expectations.ExpectColumnValuesToBeBetween(column='Price', min_value=0))
suite_sales.add_expectation(gx.expectations.ExpectColumnValuesToBeBetween(column='Revenue', min_value=0))
suite_sales.add_expectation(gx.expectations.ExpectTableRowCountToBeBetween(min_value=100000))

print('sales expectations added:', len(suite_sales.expectations))

In [ ]:
vd_sales = context.validation_definitions.add(
    gx.ValidationDefinition(name='sales_validation', data=bd_sales, suite=suite_sales)
)

results_sales = vd_sales.run(batch_parameters={'dataframe': df_sales})

print('\n--- Sales Dataset Validation ---')
print('success:', results_sales.success)

for r in results_sales.results:
    status = '[PASS]' if r.success else '[FAIL]'
    print(f'  {status}  {r.expectation_config.type}')

## 5. Validate Churn Dataset

In [ ]:
ds_churn = context.data_sources.add_pandas('churn_source')
da_churn = ds_churn.add_dataframe_asset('churn_asset')
bd_churn = da_churn.add_batch_definition_whole_dataframe('churn_batch')

suite_churn = context.suites.add(gx.ExpectationSuite(name='churn_suite'))

suite_churn.add_expectation(gx.expectations.ExpectColumnValuesToNotBeNull(column='Customer_ID'))
suite_churn.add_expectation(gx.expectations.ExpectColumnValuesToNotBeNull(column='Target_Churn'))
suite_churn.add_expectation(gx.expectations.ExpectColumnValuesToBeBetween(column='Age', min_value=18, max_value=100))
suite_churn.add_expectation(gx.expectations.ExpectColumnValuesToBeBetween(column='Satisfaction_Score', min_value=1, max_value=5))
suite_churn.add_expectation(gx.expectations.ExpectColumnValuesToBeBetween(column='Total_Spend', min_value=0))
suite_churn.add_expectation(gx.expectations.ExpectColumnValuesToBeInSet(column='Gender', value_set=['Male', 'Female', 'Other']))
suite_churn.add_expectation(gx.expectations.ExpectTableRowCountToBeBetween(min_value=500))

vd_churn = context.validation_definitions.add(
    gx.ValidationDefinition(name='churn_validation', data=bd_churn, suite=suite_churn)
)

results_churn = vd_churn.run(batch_parameters={'dataframe': df_churn})

print('\n--- Churn Dataset Validation ---')
print('success:', results_churn.success)

for r in results_churn.results:
    status = '[PASS]' if r.success else '[FAIL]'
    print(f'  {status}  {r.expectation_config.type}')

## 6. Validate Inventory Dataset

In [ ]:
ds_inv = context.data_sources.add_pandas('inventory_source')
da_inv = ds_inv.add_dataframe_asset('inventory_asset')
bd_inv = da_inv.add_batch_definition_whole_dataframe('inventory_batch')

suite_inv = context.suites.add(gx.ExpectationSuite(name='inventory_suite'))

suite_inv.add_expectation(gx.expectations.ExpectColumnValuesToNotBeNull(column='Date'))
suite_inv.add_expectation(gx.expectations.ExpectColumnValuesToNotBeNull(column='Store ID'))
suite_inv.add_expectation(gx.expectations.ExpectColumnValuesToNotBeNull(column='Product ID'))
suite_inv.add_expectation(gx.expectations.ExpectColumnValuesToBeBetween(column='Units Sold', min_value=0))
suite_inv.add_expectation(gx.expectations.ExpectColumnValuesToBeBetween(column='Inventory Level', min_value=0))
suite_inv.add_expectation(gx.expectations.ExpectColumnValuesToBeBetween(column='Price', min_value=0))
suite_inv.add_expectation(gx.expectations.ExpectColumnValuesToBeBetween(column='Discount', min_value=0, max_value=100))
suite_inv.add_expectation(gx.expectations.ExpectTableRowCountToBeBetween(min_value=10000))

vd_inv = context.validation_definitions.add(
    gx.ValidationDefinition(name='inventory_validation', data=bd_inv, suite=suite_inv)
)

results_inv = vd_inv.run(batch_parameters={'dataframe': df_inventory})

print('\n--- Inventory Dataset Validation ---')
print('success:', results_inv.success)

for r in results_inv.results:
    status = '[PASS]' if r.success else '[FAIL]'
    print(f'  {status}  {r.expectation_config.type}')

## 7. Validate RFM Output

In [ ]:
ds_rfm = context.data_sources.add_pandas('rfm_source')
da_rfm = ds_rfm.add_dataframe_asset('rfm_asset')
bd_rfm = da_rfm.add_batch_definition_whole_dataframe('rfm_batch')

suite_rfm = context.suites.add(gx.ExpectationSuite(name='rfm_suite'))

suite_rfm.add_expectation(gx.expectations.ExpectColumnValuesToNotBeNull(column='Customer ID'))
suite_rfm.add_expectation(gx.expectations.ExpectColumnValuesToBeBetween(column='Recency',   min_value=0))
suite_rfm.add_expectation(gx.expectations.ExpectColumnValuesToBeBetween(column='Frequency', min_value=1))
suite_rfm.add_expectation(gx.expectations.ExpectColumnValuesToBeBetween(column='Monetary',  min_value=0))
suite_rfm.add_expectation(gx.expectations.ExpectColumnValuesToBeBetween(column='R_score', min_value=1, max_value=5))
suite_rfm.add_expectation(gx.expectations.ExpectColumnValuesToBeBetween(column='F_score', min_value=1, max_value=5))
suite_rfm.add_expectation(gx.expectations.ExpectColumnValuesToBeBetween(column='M_score', min_value=1, max_value=5))
suite_rfm.add_expectation(gx.expectations.ExpectColumnValuesToBeInSet(
    column='Segment',
    value_set=['Champions', 'Loyal Customers', 'Potential Loyalists', 'At Risk', 'Lost']
))

vd_rfm = context.validation_definitions.add(
    gx.ValidationDefinition(name='rfm_validation', data=bd_rfm, suite=suite_rfm)
)

results_rfm = vd_rfm.run(batch_parameters={'dataframe': df_rfm})

print('\n--- RFM Output Validation ---')
print('success:', results_rfm.success)

for r in results_rfm.results:
    status = '[PASS]' if r.success else '[FAIL]'
    print(f'  {status}  {r.expectation_config.type}')

## 8. Overall Summary

In [ ]:
all_results = {
    'Sales':     results_sales,
    'Churn':     results_churn,
    'Inventory': results_inv,
    'RFM':       results_rfm,
}

print('=' * 45)
print('  DATA VALIDATION SUMMARY')
print('=' * 45)
for name, res in all_results.items():
    total  = len(res.results)
    passed = sum(1 for r in res.results if r.success)
    status = 'PASS' if res.success else 'FAIL'
    print(f'  {name:<12} {status}  ({passed}/{total} checks passed)')
print('=' * 45)

## Done!

All datasets validated. Any FAIL results need to be checked before modeling.

Next -> Day 3: Customer Segmentation (K-Means / DBSCAN)